In [1]:
import torch
from transformers import AutoTokenizer, set_seed
from parler_tts import ParlerTTSForConditionalGeneration, ParlerTTSConfig
from transformers.cache_utils import StaticCache
import soundfile as sf
import os
import gc
import torch._inductor.config

torch.set_float32_matmul_precision('high')
torch._inductor.config.triton.cudagraph_skip_dynamic_graphs = True
BATCH_SIZE = 2
# --- MONKEY PATCH FOR TRANSFORMERS 4.46.1 ---
# Forcefully add the missing max_batch_size attribute to the StaticCache class.
if not hasattr(StaticCache, "max_batch_size"):
    StaticCache.batch_size = BATCH_SIZE
    StaticCache.max_batch_size = BATCH_SIZE
# --------------------------------------------

# 1. Define Test Samples
# We pick two highly contrasting prompts to test the models' range
test_samples = [
    {
        "id": "test_1",
        "command": "change the color of upstairs attic light to springgreen.",
        "description": "Thomas speaks sadly at a slightly slow speed with high quality audio."
    },
    {
        "id": "test_2",
        "command": "change the color of upstairs attic light to springgreen.",
        "description": "Jerry speaks confusedly at a slightly fast speed with high quality audio."
    },
    {
        "id": "test_3",
        "command": "uh can you check if the kitchen bookshelf warm light is on or something",
        "description": "Elizabeth speaks happily at a quite slow speed with high quality audio."
    },
    {
        "id": "test_4",
        "command": "uh can you check if the kitchen bookshelf warm light is on or something",
        "description": "Jerry speaks neutrally at a slightly fast speed with high quality audio."
    },
]

In [2]:
output_dir = "../test/model_comparison_tests"
os.makedirs(output_dir, exist_ok=True)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. Reusable Generation Function
def run_model_test(model_id, short_name):
    print(f"\n--- Loading {model_id} ---")

    # Load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    config = ParlerTTSConfig.from_pretrained(model_id)
    config.decoder._attn_implementation = "flash_attention_2"
    model = ParlerTTSForConditionalGeneration.from_pretrained(model_id, config=config).to(device)

    max_length = 50
    compile_mode = "reduce-overhead" # "default" or "reduce-overhead"
    model.generation_config.cache_implementation = "static"
    model.forward = torch.compile(model.forward, mode=compile_mode)
    # warmup
    inputs = tokenizer(BATCH_SIZE * ["this is for compilation"], return_tensors="pt", padding="max_length", max_length=max_length).to(device)
    model_kwargs = {**inputs, "prompt_input_ids": inputs.input_ids, "prompt_attention_mask": inputs.attention_mask, }
    n_steps = 1 if compile_mode == "default" else 2
    for _ in range(n_steps):
        _ = model.generate(**model_kwargs)

    # Lock the seed so both models generate from the exact same random starting point
    set_seed(42)

    # Process in batches
    for i in range(0, len(test_samples), BATCH_SIZE):
        batch_samples = test_samples[i:i+BATCH_SIZE]
        actual_batch_size = len(batch_samples)

        descriptions = [s["description"] for s in batch_samples]
        commands = [s["command"] for s in batch_samples]

        if actual_batch_size < BATCH_SIZE:
            pad_amount = BATCH_SIZE - actual_batch_size
            descriptions.extend(["dummy"] * pad_amount)
            commands.extend(["dummy"] * pad_amount)

        print(f"Generating batch {i//BATCH_SIZE + 1} with {short_name}...")

        # Tokenize inputs
        inputs = tokenizer(descriptions, return_tensors="pt", padding="max_length", max_length=max_length).to(device)
        prompt = tokenizer(commands, return_tensors="pt", padding="max_length", max_length=max_length).to(device)

        # Generate audio
        generation = model.generate(input_ids=inputs.input_ids,
                                    attention_mask=inputs.attention_mask,
                                    prompt_input_ids=prompt.input_ids,
                                    prompt_attention_mask=prompt.attention_mask,
                                    return_dict_in_generate=True,
                                    max_new_tokens=860,
                                    do_sample=True, # Enable sampling for more natural variance
                                    temperature=1.05, # Increase temperature for more creative generation
                                    repetition_penalty=1.0, # Penalize repetition for more natural sounding text
                                    top_p=0.8, # Nucleus sampling to focus on top % of probability mass
                                    )

        for j in range(actual_batch_size):
            sample = batch_samples[j]
            audio_len = generation.audios_length[j]
            audio_arr = generation.sequences[j, :audio_len].cpu().numpy().squeeze()

            # Save file
            filename = f"{sample['id']}_{short_name}.wav"
            filepath = os.path.join(output_dir, filename)
            sf.write(filepath, audio_arr, model.config.sampling_rate)

    print(f"Finished {short_name}. Clearing VRAM...")

    # 3. Aggressive Memory Cleanup to prevent OOM errors
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

Using device: cuda:0


In [3]:
# 3. Execute Tests Sequentially
# Test the Expresso-tuned mini model
run_model_test("parler-tts/parler-tts-mini-expresso", "MINI_EXPRESSO")


--- Loading parler-tts/parler-tts-mini-expresso ---


You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
/home/vito/miniconda3/envs/pytorch/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Config of the text_encoder: <class 'transformers.models.t5.modeling_t5.T5EncoderModel'> is overwritten by shared text_encoder config: T5Config {
  "_name_or_path": "google/flan-t5-base",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num

Generating batch 1 with MINI_EXPRESSO...
Generating batch 2 with MINI_EXPRESSO...
Finished MINI_EXPRESSO. Clearing VRAM...
